In [ ]:
!pip install -q crewai
!pip install -q openai
!pip install -q unstructured
!pip install -q tools
!pip install -q tenacity==8.3.0
!pip install -q langchain
!pip install -q langchain_groq
!pip install -q cohere
!pip install -q langchain_community
!pip install -q 'crewai[tools]'

In [ ]:
from crewai import Agent, Task, Crew
from langchain_community.chat_models import ChatCohere
from langchain_openai import OpenAI
from langchain_groq import ChatGroq


In [ ]:
#warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os

COHERE_API_KEY="WmDvMVZKTaO2N33JRmEIC9WtBiRG7kDsu5ZiyUJU"
OPENAI_API_KEY="sk-crew-ai-gqQfhuNQaIn8AKoIRg8UT3BlbkFJfEam04Z3SQEPwjWabC4Y"
GROQ_API_KEY="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"
SERPER_API_KEY= "25b084c4ee79d46c1866395746dbdf145c1729d9"


os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['COHERE_API_KEY'] = COHERE_API_KEY
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['SERPER_API_KEY'] = SERPER_API_KEY

#Tools
from crewai_tools import (
    SerperDevTool,
    WebsiteSearchTool
)

search_tool = SerperDevTool()
web_search_tool = WebsiteSearchTool()

#LLMs

cohere = ChatCohere(cohere_api_key=COHERE_API_KEY,
                    temperaature= 0.3)
openai = OpenAI(api_key=OPENAI_API_KEY)
groq = ChatGroq(
                temperature=0,
                groq_api_key=GROQ_API_KEY,
                model_name="mixtral-8x7b-32768"
            )


In [ ]:
#Agent
 cold_email_reviewer = Agent(
                         role='Cold Email Reviewer',
                         goal="""Review the generated {cold_email} to ensure they follow the format: 'Title: Painpoint: Job title: email:' for a total of five emails.""",
                         backstory="""You are responsible for reviewing {cold_email} to ensure they adhere to a specific format. This format is 'Title: Painpoint: Job title: email:'.
                                      Your job is to ensure that the generated {cold_email} are properly formatted and meet this requirement for a total of five emails.""",
                        allow_delegation=False,
                        verbose=True,
                        llm=groq,
                        max_iter= 50
                       )

In [ ]:
coldEmailReview = Task (
       description= """
                       Review {cold_email} to ensure they follow the required format: 'Title: Painpoint: Job title: email: where email is not the email address but the cold email'.
              Make sure that the format is strictly adhered to for a total of five emails.
              If there are any deviations from the format, provide detailed feedback on what needs to be corrected.
              The task involves carefully checking each email to ensure that it meets the format requirements and that the content is clear and properly structured.
              The final output should include feedback on the email format adherence and any necessary corrections.
                   """,
            expected_output="Five cold emails, that adhere to the format 'Title: Painpoint: Job title: email:'.",
            agent=cold_email_generator)

In [ ]:
#Crew
crew = Crew(
    agents=[cold_email_generator],
    tasks=[cold_email_reviewer],
    verbose=True,
)

In [ ]:
#Execute Crew
result = crew.kickoff(inputs={
    "cold_email": """
                      """

     })

In [ ]:
from IPython.display import Markdown
Markdown(result)